# Geopack Python SDK: Getting Started

Welcome to the Geopack Geoportal v2 Python SDK. This notebook will guide you through the basics of connecting to the portal, authenticating, and exploring available datasets.

---

### 🚀 Quick Setup

#### 1. Environment & Dependencies
You have two ways to prepare your environment:

*   **Option A: Inside Notebook (Easiest)**
    Run the cell below (under "Setup Dependencies") to install everything directly into your current kernel.
*   **Option B: Manual Virtual Environment (Recommended for Clean Setup)**
    Open your terminal in the `python-sdk` folder and run:
    ```bash
    python -m venv venv
    venv\Scripts\activate  # On Windows
    source venv/bin/activate  # On Linux/macOS
    pip install geopandas matplotlib python-dotenv
    ```

#### 2. SDK Installation Options
*   **Method A (Local Source)**: This notebook is pre-configured to use the `src/` folder directly. No installation needed if you are in the repository!
*   **Method B (Registry)**: `pip install geopack-sdk`

#### 3. Credentials
Ensure you have a `.env` file in this `notebooks/` folder with your `GEOPACK_API_URL`, `GEOPACK_USERNAME`, and `GEOPACK_PASSWORD`.

---

In [1]:
# Setup Dependencies
# Run this cell to install required libraries if you haven't already
%pip install python-dotenv geopandas matplotlib



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Initialize the Client
First, we import the `GeopackClient` and initialize it with the API URL.

In [2]:
# 1. Setup & SDK Import
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import os
import sys
from dotenv import load_dotenv

# --- SMART SOURCE IMPORT ---
# This ensures we use the local 'src' directory even if the package isn't installed.
# It calculates the path relative to this notebook's location.
try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    
    if os.path.exists(source_path):
        if source_path not in sys.path:
            sys.path.insert(0, source_path)
        print(f"ℹ️ Using SDK from local source: {source_path}")
    else:
        print("ℹ️ Using SDK from installed site-packages (pip)")
except Exception:
    print("⚠️ Could not determine local source path, falling back to pip.")

from geopack_sdk import (
    GeopackClient,
    GeopackAPIError,
    GeopackAuthError,
    GeopackError,
    GeopackTaskError,
    GeopackTimeoutError,
)

# --- Load Configuration ---
load_dotenv()
API_URL = os.getenv("GEOPACK_API_URL", "http://localhost:3000/api")
client = GeopackClient(base_url=API_URL)

print(f"✅ Client initialized for: {API_URL}")



ℹ️ Using SDK from local source: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\src
✅ Client initialized for: http://localhost:3000/api


## 2. Authentication
Log in using your credentials to obtain a JWT token.

In [10]:
import os
USERNAME = os.getenv("GEOPACK_USERNAME", "admin")
PASSWORD = os.getenv("GEOPACK_PASSWORD", "password")

try:
    client.auth.login(USERNAME, PASSWORD)
    print("✅ Login successful!")
except GeopackAuthError as e:
    print(f"❌ Authentication failed (HTTP {e.status_code}): {e.message}")
except GeopackAPIError as e:
    print(f"❌ API error during login (HTTP {e.status_code}): {e.message}")
except GeopackError as e:
    print(f"❌ Login failed: {e.message}")



✅ Login successful!


## Handling SDK Errors

The SDK raises typed exceptions instead of generic `Exception`. Use them to branch on auth, API, task, and timeout failures.


In [11]:
def report_sdk_error(context: str, exc: Exception) -> None:
    """Print a readable message for Geopack SDK exceptions."""
    if isinstance(exc, GeopackAuthError):
        print(f"[{context}] Auth error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackAPIError):
        print(f"[{context}] API error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackTaskError):
        print(f"[{context}] Task {exc.task_id} {exc.status}: {exc.message}")
    elif isinstance(exc, GeopackTimeoutError):
        print(f"[{context}] Timeout: {exc.message}")
    elif isinstance(exc, GeopackError):
        print(f"[{context}] {exc.message}")
    else:
        print(f"[{context}] Unexpected: {exc}")


# Example: not-found dataset -> GeopackAPIError (often 404)
try:
    client.datasets.get(999999999)
except GeopackAPIError as e:
    report_sdk_error("datasets.get", e)
    print(f"  status_code={e.status_code}")



[datasets.get] API error (HTTP 404): Dataset not found or access denied.
  status_code=404


## 3. Explore Datasets
Now let's list the available datasets in the portal.

In [12]:
datasets = client.datasets.list(page_size=50)

print(f"Found {len(datasets.datasets)} datasets:\n")
for ds in datasets.datasets:
    print(f"- [{ds.id}] {ds.name}  ({ds.dataType or 'N/A'} {ds.subType or ''}) [{ds.dataStore.name if ds.dataStore else 'N/A'}]")


Found 50 datasets:

- [2407] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2406] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2405] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2404] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2360] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2343] hillshade12  (raster MBTiles) [Default Filesystem GDB]
- [2311] LC80120292013170LGN00_NDVI.tif  (raster MBTiles) [Default Filesystem GDB]
- [2308] LC80120292013170LGN00.tif  (raster MBTiles) [Default Filesystem GDB]
- [2307] LC80120292013170LGN00.tif  (raster MultiBandGeneric) [pg_test]
- [2304] xyzdem-3857  (raster MBTiles) [Default Filesystem GDB]
- [2303] xyzdem  (raster MBTiles) [Default Filesystem GDB]
- [2302] xyzdem  (raster SingleBand) [pg_test]
- [2294] xytable  (table ) [pg_test]
- [2293] مکانیابی احداث پارکینگ دوچرخه  (vector MultiPolygon) [Default Filesystem GDB]
- [2292] نقشه رستری مناسبت 

## 4. Advanced Filtering
The SDK supports the same powerful filtering as the portal. Let's find only the **Vector** datasets.

In [13]:
# Using the same filter structure as the Portal UI
vector_only = client.datasets.list(active_filters={"dataType": "vector"})

print(f"Found {len(vector_only.datasets)} vector datasets.")


Found 10 vector datasets.


## 5. Get Detailed Metadata
Retrieve full information about a specific dataset, including its SRS, extent, and statistics.

In [14]:
if datasets.datasets:
    target_id = datasets.datasets[0].id
    details = client.datasets.get(target_id)
    
    import json
    print(f"Detailed info for dataset #{target_id}:")
    print(json.dumps(details.model_dump(mode='json'), indent=2, default=str))
else:
    print("No datasets available to inspect.")


Detailed info for dataset #2407:
{
  "id": 2407,
  "name": "Rural_District.shp",
  "description": "Dataset from file: Rural_District.shp.geojson",
  "dataType": "vector",
  "subType": "MultiPolygon",
  "keywords": "",
  "ownerUserId": 1,
  "workgroupId": 1,
  "dataStoreId": 11,
  "details": "{\"type\":\"vector\",\"source\":{\"driver\":\"MVT\",\"container\":\"app_data/gdb/default\",\"datasetFolder\":\"f644d7b8-8a18-48d8-9538-cfb67de44cd9\",\"originalFileName\":\"Rural_District.shp.geojson\",\"fileType\":\".geojson\",\"geojsonPath\":\"original\\\\original.geojson\",\"isReadonly\":true},\"tableName\":\"f644d7b8-8a18-48d8-9538-cfb67de44cd9\",\"shapeField\":\"geom\",\"spatialReference\":{\"srid\":4326,\"wkt\":\"GEOGCRS[\\\"WGS 84\\\",\\n    ENSEMBLE[\\\"World Geodetic System 1984 ensemble\\\",\\n        MEMBER[\\\"World Geodetic System 1984 (Transit)\\\"],\\n        MEMBER[\\\"World Geodetic System 1984 (G730)\\\"],\\n        MEMBER[\\\"World Geodetic System 1984 (G873)\\\"],\\n        MEMB